# QuickCart Market Basket Analysis




In [ ]:
import pandas as pd
from itertools import combinations
from io import StringIO

dim_skus = pd.read_csv(StringIO(sku_csv))

frequent_itemsets.head()

,support,itemsets
0,0.0568,{'Air Freshener'}
1,0.0608,{'Apple 1kg'}
2,0.0600,{'Banana 1kg'}
3,0.0578,{'Banana Chips 150g'}
4,0.0604,{'Bar Soap 125g'}


In [ ]:
# Convert the stored itemset text into Python frozensets
def parse_itemset(value):
    return frozenset(eval(value, {"frozenset": frozenset}))

frequent_itemsets["itemsets"] = frequent_itemsets["itemsets"].apply(parse_itemset)

print("Frequent itemsets:", len(frequent_itemsets))
print("SKU catalog:", len(dim_skus))
print("1-itemsets:", sum(len(x) == 1 for x in frequent_itemsets["itemsets"]))
print("2-itemsets:", sum(len(x) == 2 for x in frequent_itemsets["itemsets"]))
print("3-itemsets:", sum(len(x) == 3 for x in frequent_itemsets["itemsets"]))
print("4-itemsets:", sum(len(x) == 4 for x in frequent_itemsets["itemsets"]))
print("5-itemsets:", sum(len(x) == 5 for x in frequent_itemsets["itemsets"]))

Frequent itemsets: 256
SKU catalog: 60
1-itemsets: 60
2-itemsets: 78
3-itemsets: 86
4-itemsets: 30
5-itemsets: 2


## Association-rule generation




In [ ]:
support_map = {frozenset(row.itemsets): row.support for row in frequent_itemsets.itertuples()}

rules = []
for itemset, itemset_support in support_map.items():
    if len(itemset) < 2:
        continue
    items = list(itemset)
    for r in range(1, len(items)):
        for antecedent_items in combinations(items, r):
            antecedent = frozenset(antecedent_items)
            consequent = itemset - antecedent
            if antecedent in support_map and consequent in support_map:
                confidence = itemset_support / support_map[antecedent]
                lift = confidence / support_map[consequent]
                rules.append({
                    "antecedents": ", ".join(sorted(antecedent)),
                    "consequents": ", ".join(sorted(consequent)),
                    "support": itemset_support,
                    "confidence": confidence,
                    "lift": lift
                })

rules = pd.DataFrame(rules).sort_values("lift", ascending=False).reset_index(drop=True)
print("Generated rules:", len(rules))
rules.head(10)

Generated rules: 1152


,antecedents,consequents,support,confidence,lift
0,"Hand Sanitizer 200ml, Instant Coffee 100g","Choco Cookies 150g, Face Wash 100ml",0.0206,0.735714,26.850886
1,"Choco Cookies 150g, Face Wash 100ml","Hand Sanitizer 200ml, Instant Coffee 100g",0.0206,0.751825,26.850886
2,"Face Wash 100ml, Instant Coffee 100g","Choco Cookies 150g, Hand Sanitizer 200ml",0.0206,0.741007,26.848087
3,"Choco Cookies 150g, Hand Sanitizer 200ml","Face Wash 100ml, Instant Coffee 100g",0.0206,0.746377,26.848087
4,"Choco Cookies 150g, Poha 500g","Ghee 500ml, Instant Coffee 100g",0.0234,0.774834,26.001156
5,"Ghee 500ml, Instant Coffee 100g","Choco Cookies 150g, Poha 500g",0.0234,0.785235,26.001156
6,"Choco Cookies 150g, Ghee 500ml","Instant Coffee 100g, Poha 500g",0.0234,0.759740,25.324675
7,"Instant Coffee 100g, Poha 500g","Choco Cookies 150g, Ghee 500ml",0.0234,0.780000,25.324675
8,"Ghee 500ml, Toned Milk 1L","Poha 500g, White Bread 400g",0.0292,0.802198,21.798853
9,"Poha 500g, White Bread 400g","Ghee 500ml, Toned Milk 1L",0.0292,0.793478,21.798853


In [ ]:
# Actionable rules: reasonable support and strong lift
actionable = rules[(rules["support"] > 0.05) & (rules["lift"] > 2)].copy()
actionable["support_pct"] = actionable["support"] * 100
actionable["confidence_pct"] = actionable["confidence"] * 100

actionable[["antecedents","consequents","support_pct","confidence_pct","lift"]].head(20)

,antecedents,consequents,support_pct,confidence_pct,lift
186,Ghee 500ml,Poha 500g,14.68,86.049238,5.067682
187,Poha 500g,Ghee 500ml,14.68,86.454653,5.067682
192,Face Wash 100ml,Hand Sanitizer 200ml,13.18,83.949045,5.051086
193,Hand Sanitizer 200ml,Face Wash 100ml,13.18,79.302046,5.051086
292,Instant Coffee 100g,Choco Cookies 150g,15.20,85.875706,4.835344
293,Choco Cookies 150g,Instant Coffee 100g,15.20,85.585586,4.835344
352,Cola 750ml,Potato Chips 90g,19.74,91.219963,4.176738
353,Potato Chips 90g,Cola 750ml,19.74,90.384615,4.176738
364,"Cola 750ml, Toned Milk 1L",Potato Chips 90g,5.18,90.877193,4.161044
365,Potato Chips 90g,"Cola 750ml, Toned Milk 1L",5.18,23.717949,4.161044


## Locked pattern verification

In [ ]:
locked_patterns = [
    ("Toned Milk 1L", "White Bread 400g", 0.2384, 0.9386, 3.69),
    ("Basmati Rice 5kg", "Toor Dal 1kg", 0.1534, 0.7740, 3.74),
    ("Potato Chips 90g", "Cola 750ml", 0.1974, 0.9038, 4.18),
    ("Ghee 500ml", "Poha 500g", 0.1468, 0.8605, 5.07),
    ("Instant Coffee 100g", "Choco Cookies 150g", 0.1520, 0.8588, 4.84),
    ("Hand Sanitizer 200ml", "Face Wash 100ml", 0.1318, 0.7930, 5.05),
]

verification = []
for a, b, exp_s, exp_c, exp_l in locked_patterns:
    row = rules[
        (rules["antecedents"] == a) &
        (rules["consequents"] == b)
    ].iloc[0]
    verification.append({
        "Rule": f"{a} → {b}",
        "Support": row["support"],
        "Confidence": row["confidence"],
        "Lift": row["lift"],
        "Matches locked answer": (
            abs(row["support"]-exp_s) < 1e-6 and
            abs(row["confidence"]-exp_c) < 1e-3 and
            abs(row["lift"]-exp_l) < 0.01
        )
    })

verification = pd.DataFrame(verification)
verification

,Rule,Support,Confidence,Lift,Matches locked answer
0,Toned Milk 1L → White Bread 400g,0.2384,0.938583,3.686499,True
1,Basmati Rice 5kg → Toor Dal 1kg,0.1534,0.773966,3.742581,True
2,Potato Chips 90g → Cola 750ml,0.1974,0.903846,4.176738,True
3,Ghee 500ml → Poha 500g,0.1468,0.860492,5.067682,True
4,Instant Coffee 100g → Choco Cookies 150g,0.1520,0.858757,4.835344,True
5,Hand Sanitizer 200ml → Face Wash 100ml,0.1318,0.793020,5.051086,True


## Business recommendations




In [ ]:
# Strong, reasonably supported rules for recommendation placement
recommendations = actionable[
    (actionable["confidence"] >= 0.75)
].sort_values(["lift","support"], ascending=[False,False])

recommendations[
    ["antecedents","consequents","support_pct","confidence_pct","lift"]
].head(15)

,antecedents,consequents,support_pct,confidence_pct,lift
186,Ghee 500ml,Poha 500g,14.68,86.049238,5.067682
187,Poha 500g,Ghee 500ml,14.68,86.454653,5.067682
192,Face Wash 100ml,Hand Sanitizer 200ml,13.18,83.949045,5.051086
193,Hand Sanitizer 200ml,Face Wash 100ml,13.18,79.302046,5.051086
292,Instant Coffee 100g,Choco Cookies 150g,15.20,85.875706,4.835344
293,Choco Cookies 150g,Instant Coffee 100g,15.20,85.585586,4.835344
352,Cola 750ml,Potato Chips 90g,19.74,91.219963,4.176738
353,Potato Chips 90g,Cola 750ml,19.74,90.384615,4.176738
364,"Cola 750ml, Toned Milk 1L",Potato Chips 90g,5.18,90.877193,4.161044
375,"Cola 750ml, White Bread 400g",Potato Chips 90g,5.14,90.492958,4.143450
